In [ ]:
# DML Bates Surrogate Training Pipeline (100k samples, GPU)
# Automatically generated for Kaggle - push with:
#   kaggle kernels push -p .
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.func import vmap, jacrev
import time
from scipy.stats import qmc
import multiprocessing as mp


In [ ]:
# --- bates_fft.py ---
import numpy as np

def bates_cf(u, S0, r, q, T, v0, kappa, theta, sigma, rho, lambda_, muJ, vJ):
    """
    Characteristic function of the Bates model.
    """
    i = 1j
    d = np.sqrt((kappa - i*rho*sigma*u)**2 + (u**2 + i*u)*sigma**2)
    g = (kappa - i*rho*sigma*u - d) / (kappa - i*rho*sigma*u + d)

    cf_heston = np.exp(i*u*(np.log(S0) + (r - q)*T)) * \
                np.exp((kappa*theta)/(sigma**2) * ((kappa - i*rho*sigma*u - d)*T - 2*np.log((1 - g*np.exp(-d*T))/(1 - g)))) * \
                np.exp((v0/sigma**2) * (kappa - i*rho*sigma*u - d) * (1 - np.exp(-d*T))/(1 - g*np.exp(-d*T)))

    omega = -lambda_ * (np.exp(muJ + 0.5 * vJ**2) - 1)
    cf_jump = np.exp(lambda_ * T * (np.exp(i*u*muJ - 0.5 * u**2 * vJ**2) - 1) + i*u*omega*T)

    return cf_heston * cf_jump

def carr_madan_fft(S0, r, q, T, v0, kappa, theta, sigma, rho, lambda_, muJ, vJ, N=4096, eta=0.1, alpha=1.5):
    """
    Carr-Madan FFT option pricing for Bates model.
    """
    v = np.arange(N) * eta
    i = 1j

    def psi(v_val):
        u = v_val - (alpha + 1)*i
        cf = bates_cf(u, S0, r, q, T, v0, kappa, theta, sigma, rho, lambda_, muJ, vJ)
        return np.exp(-r*T) * cf / (alpha**2 + alpha - v_val**2 + i*(2*alpha + 1)*v_val)

    psi_v = psi(v)

    weights = np.ones(N)
    weights[1::2] = 4
    weights[2::2] = 2
    weights[0] = 1
    weights[-1] = 1
    weights = weights / 3

    lambda_grid = 2 * np.pi / (N * eta)
    b = (N * lambda_grid) / 2
    ku = -b + lambda_grid * np.arange(N)
    strikes = np.exp(ku)

    x = np.exp(i * b * v) * psi_v * eta * weights
    x[0] = x[0] * 0.5
    y = np.fft.fft(x)
    call_prices = (np.exp(-alpha * ku) / np.pi) * np.real(y)
    return strikes, call_prices

def price_bates_surface(S0, r, q, maturities, moneyness_grid, params):
    """
    Prices for a grid of maturities and moneyness via FFT + interpolation.
    params = [v0, kappa, theta, sigma, rho, lambda_, muJ, vJ]
    """
    v0, kappa, theta, sigma, rho, lambda_, muJ, vJ = params
    prices = np.zeros((len(maturities), len(moneyness_grid)))
    for i, T in enumerate(maturities):
        strikes, calls = carr_madan_fft(S0, r, q, T, v0, kappa, theta, sigma, rho, lambda_, muJ, vJ)
        prices[i, :] = np.interp(moneyness_grid * S0, strikes, calls)

    # Carr-Madan truncation error can dip slightly below zero on the
    # short-maturity deep-OTM wing; floor at zero (no-arbitrage).
    return np.maximum(prices, 0.0)


In [ ]:
# --- generate_dataset.py --- (10D: 8 Bates params + r + q)
def _price_one_sample(args):
    """Worker for multiprocessing: prices + central-difference Greeks for one 10D sample."""
    params, maturities, moneyness_grid, h = args
    S0 = 1.0
    p8 = params[:8]
    r_val, q_val = params[8], params[9]
    prices = price_bates_surface(S0, r_val, q_val, maturities, moneyness_grid, p8).flatten()
    grads = np.zeros((prices.shape[0], 10))
    for j in range(10):
        pp = list(params); pp[j] += h
        pm = list(params); pm[j] -= h
        pr_plus = price_bates_surface(S0, pp[8], pp[9], maturities, moneyness_grid, pp[:8]).flatten()
        pr_minus = price_bates_surface(S0, pm[8], pm[9], maturities, moneyness_grid, pm[:8]).flatten()
        grads[:, j] = (pr_plus - pr_minus) / (2 * h)
    return prices, grads

def generate_dataset(num_samples=100000, save_path='dataset_dml.pt', seed=42, n_workers=4):
    """
    10D inputs: [v0, kappa, theta, sigma, rho, lambda_, muJ, vJ, r, q].
    n_workers>1 parallelizes FFT generation across CPU cores (Kaggle gives 4).
    """
    np.random.seed(seed)
    maturities = np.array([30/365, 60/365, 90/365, 120/365, 150/365])
    moneyness_grid = np.linspace(0.8, 1.2, 11)
    num_prices = len(maturities) * len(moneyness_grid)

    sampler = qmc.LatinHypercube(d=10, seed=seed)
    l_bounds = [0.01, 0.5, 0.01, 0.1, -0.9, 0.0, -0.2, 0.05, 0.0, 0.0]
    u_bounds = [0.2, 5.0, 0.2, 0.8, 0.0, 1.0, 0.0, 0.3, 0.1, 0.05]
    X_samples = qmc.scale(sampler.random(n=num_samples), l_bounds, u_bounds)

    print(f"Generating {num_samples} samples with Greeks (DML), n_workers={n_workers}...")
    start_time = time.time()
    h = 1e-4
    tasks = [(X_samples[i].tolist(), maturities, moneyness_grid, h) for i in range(num_samples)]

    X = X_samples
    Y = np.zeros((num_samples, num_prices))
    dY_dX = np.zeros((num_samples, num_prices, 10))
    if n_workers > 1:
        with mp.Pool(n_workers) as pool:
            results = pool.map(_price_one_sample, tasks, chunksize=32)
    else:
        results = [_price_one_sample(t) for t in tasks]
    for i, (prices, grads) in enumerate(results):
        Y[i, :] = prices
        dY_dX[i, :, :] = grads
        if (i + 1) % 5000 == 0:
            print(f"Generated {i+1}/{num_samples} in {time.time() - start_time:.1f}s")

    torch.save({
        'X': torch.tensor(X, dtype=torch.float32),
        'Y': torch.tensor(Y, dtype=torch.float32),
        'dY_dX': torch.tensor(dY_dX, dtype=torch.float32),
        'maturities': maturities,
        'moneyness': moneyness_grid,
    }, save_path)
    print(f"DML Dataset saved to {save_path} ({time.time() - start_time:.1f}s total)")


In [ ]:
# --- model.py ---
import torch
import torch.nn as nn

class BatesNeuralSurrogate(nn.Module):
    def __init__(self, input_dim=10, output_dim=55):
        super(BatesNeuralSurrogate, self).__init__()
        # ELU activation: smooth first derivatives, required for Sobolev training
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ELU(),
            nn.Linear(128, 256),
            nn.ELU(),
            nn.Linear(256, 256),
            nn.ELU(),
            nn.Linear(256, 128),
            nn.ELU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# --- train_dml.py --- (Sobolev training, OTM-weighted price loss)
def train_dml_surrogate(dataset_path='dataset_100000_dml.pt', epochs=100, batch_size=512, alpha=1.0, seed=42):
    """
    epochs=100 with 100k samples / batch 512 gives ~19.6k optimizer steps -
    more total steps AND more data diversity than the 1k x 500-epoch run,
    while finishing comfortably inside a Kaggle GPU session.
    """
    torch.manual_seed(seed)
    device = torch.device('cpu')
    print(f"Using device: {device}")

    data = torch.load(dataset_path, weights_only=False)
    X = data['X'].to(device)
    Y = data['Y'].to(device)
    dY_dX = data['dY_dX'].to(device)

    X_mean = X.mean(dim=0); X_std = X.std(dim=0)
    Y_mean = Y.mean(dim=0); Y_std = Y.std(dim=0)
    X_scaled = (X - X_mean) / X_std
    Y_scaled = (Y - Y_mean) / Y_std

    # Chain rule: d(Y_scaled)/d(X_scaled) = (dY/dX) * (X_std / Y_std)
    dY_dX_scaled = dY_dX * (X_std.view(1, 1, -1) / Y_std.view(1, -1, 1))

    dataset = TensorDataset(X_scaled, Y_scaled, dY_dX_scaled)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            pin_memory=False)

    model = BatesNeuralSurrogate(input_dim=X.shape[1], output_dim=Y.shape[1]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    def predict(x_single):
        return model(x_single.unsqueeze(0)).squeeze(0)
    compute_batch_jacobian = vmap(jacrev(predict))

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    best_loss = float('inf')

    print("Starting Differential Machine Learning (Sobolev) training...")
    start_time = time.time()

    for epoch in range(epochs):
        epoch_loss = 0.0; price_sum = 0.0; greeks_sum = 0.0
        for batch_X, batch_Y, batch_dY in dataloader:
            batch_X = batch_X.to(device, non_blocking=True)
            batch_Y = batch_Y.to(device, non_blocking=True)
            batch_dY = batch_dY.to(device, non_blocking=True)
            optimizer.zero_grad()

            outputs = model(batch_X)
            pred_dY = compute_batch_jacobian(batch_X)

            # OTM-weighted price loss: weights from UNSCALED positive prices
            # (batch_Y is z-scored and can be negative - dividing by it would
            # produce negative weights and corrupt the loss).
            prices_unscaled = batch_Y * Y_std + Y_mean
            w = 1.0 / (prices_unscaled + 0.01)
            w = w / w.mean()  # keep loss scale comparable across batches
            loss_price = torch.mean(w * (outputs - batch_Y) ** 2)
            loss_greeks = torch.nn.functional.mse_loss(pred_dY, batch_dY)

            loss = loss_price + alpha * loss_greeks
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * batch_X.size(0)
            price_sum += loss_price.item() * batch_X.size(0)
            greeks_sum += loss_greeks.item() * batch_X.size(0)

        n = len(dataloader.dataset)
        epoch_loss /= n; price_sum /= n; greeks_sum /= n
        scheduler.step(epoch_loss)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save({'model_state_dict': model.state_dict(),
                        'X_mean': X_mean, 'X_std': X_std,
                        'Y_mean': Y_mean, 'Y_std': Y_std}, 'bates_surrogate_dml.pth')

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Total: {epoch_loss:.6f} | Price: {price_sum:.6f} | Greeks: {greeks_sum:.6f} | {time.time()-start_time:.0f}s")

    print(f"Training complete in {time.time() - start_time:.2f}s. Best model saved to 'bates_surrogate_dml.pth'.")


In [ ]:
# RUN PIPELINE
# Estimated on Kaggle (4 CPU workers + T4 GPU): ~2-4h generation + ~1-2h training.
if __name__ == '__main__':
    generate_dataset(num_samples=100000, save_path='dataset_100000_dml.pt', n_workers=4)
    train_dml_surrogate(dataset_path='dataset_100000_dml.pt', epochs=100, batch_size=512)
